In [2]:
import pandas as pd
from lxml import etree
from bs4 import BeautifulSoup
import requests,re

In [42]:
model = 'Claude'
soup = BeautifulSoup(requests.get('https://claude.com/pricing').text,'html.parser')
dom = etree.HTML(str(soup))

Plans = [i.text.strip() for i in dom.xpath('//*[@id="main"]/section/div[3]/div[2]/div[4]/div[1]/div/div[1]/div[2]/div/div/div[*]/div[2]/h3')]   # all types of Plans

details_html = soup.find_all("div", class_="card_pricing_list_wrap")[:-2]
details = []  # This will hold all plans Details

for section in details_html:
    features_list = []
    title_div = section.find("div", class_="card_pricing_list_title")
    title = title_div.get_text(strip=True)[:-1].replace(', plus',' +') if title_div else None
    features_list.append(title)
    features_ul = section.find("ul")
    if features_ul:
        for li in features_ul.find_all("li"):
            features_list.append(li.get_text(strip=True))
    features_list = [x for x in features_list if x is not None]
    details.append(features_list)

plan_duration = [i.text.strip() for i in dom.xpath('//*[@id="main"]/section/div[3]/div[2]/div[4]/div[1]/div/div[1]/div[2]/div/div/div[*]/div[3]/div[2]/div/p')]
prices = [int(re.sub(r'\D', '', i)) for i in dom.xpath('//*[@id="main"]/section/div[3]/div[2]/div[4]/div[1]/div/div[1]/div[2]/div/div/div[*]/div[3]/div[1]/text()')]
claud_df = pd.DataFrame([Plans,prices,details,plan_duration]).T
claud_df.columns = ['Plan','Amount','Details','Time Period']
claud_df.insert(0,'Model','Claude')

new_rows = []
rows_to_drop = []

for idx, row in claud_df.iterrows():
    if row['Plan'] == 'Pro':
        text = row['Time Period']
        # extract prices
        prices = re.findall(r'\$(\d+)', text)
        if len(prices) >= 2:
            yearly_price = int(prices[0])   # $17
            monthly_price = int(prices[1])  # $20
            # yearly plan
            new_rows.append({'Model': row['Model'],'Plan': 'Pro','Amount': yearly_price,'Details': row['Details'],'Time Period': 'Annually'})

            # monthly plan
            new_rows.append({'Model': row['Model'],'Plan': 'Pro','Amount': monthly_price,'Details': row['Details'],'Time Period': 'Monthly'})
        rows_to_drop.append(idx)

# drop original Pro row
claud_df = claud_df.drop(rows_to_drop)
claud_df = pd.concat([claud_df, pd.DataFrame(new_rows)], ignore_index=True)
claud_df.loc[claud_df['Time Period'].str.lower().str.contains('free for'), 'Time Period'] = 'Free'
claud_df.loc[claud_df['Time Period'].str.lower().str.contains('monthly'), 'Time Period'] = 'Monthly'
# claud_df.to_csv('qwerty.csv',index=False)

In [48]:
claud_df

,Model,Plan,Amount,Details,Time Period
0,Claude,Free,0,"[Chat on web, iOS, Android, and on your deskto...",Free
1,Claude,Max,100,"[Everything in Pro +, Choose 5x or 20x more us...",Monthly
2,Claude,Pro,200,"[Everything in Free +, More usage*, Includes C...",Annually
3,Claude,Pro,20,"[Everything in Free +, More usage*, Includes C...",Monthly


In [67]:
model = 'Cursor'
soup = BeautifulSoup(requests.get('https://cursor.com/pricing').text,'html.parser').find('div',class_='gap-g1 grid grid-cols-1 md:grid-cols-2 lg:grid-cols-4')
dom = etree.HTML(str(soup))

In [ ]:
details = [[j.strip() for j in i.find('ul',class_='mt-v9/12 space-y-v2/12').text.split('✓')][1:] for i in soup.find_all('div',class_='col-span-full row-span-full flex flex-col justify-between')]

[['No credit card required',
  'Limited Agent requests',
  'Limited Tab completions'],
 ['Extended limits on Agent',
  'Unlimited Tab completions',
  'Cloud Agents',
  'Maximum context windows'],
 ['3x usage on all OpenAI, Claude, Gemini models'],
 ['20x usage on all OpenAI, Claude, Gemini models',
  'Priority access to new features']]

In [ ]:
[i.text.strip() for i in dom.xpath('//*[@id="main"]/section[1]/div/div[2]/div/div/a[*]/div/div[1]/p[1]/span[1]')]

[]

In [86]:
urls = ['new-team?yearly=true&_rsc=1s7iu',
        'checkoutDeepControl?tier=ultra&yearly=true&_rsc=1s7iu',
        'checkoutDeepControl?tier=pro_plus&yearly=true&_rsc=1s7iu',
        'checkoutDeepControl?yearly=true&_rsc=1s7iu',]

In [88]:
html = []

for url in urls:
    html.append(BeautifulSoup(requests.get(f'https://cursor.com/api/auth/{url}').content,'html.parser'))